# clue_runs

> Runs on the Phase 11 clue board: set-up of the board and its channel, the information metrics (shares, redundant and false shares, assisted solves, share-to-use time, pooled-bound share, rewards of sharers and non-sharers) with team totals, rule-agent and language-model runs, and the dispatcher for `runWorkload`.

In [ ]:
#| default_exp clue_runs

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np

from explRL.clue_agents import ClueBoard, ClueLead, ClueLiar, ClueMessages, RuleAgent
from explRL.clue_text import TextClueEnv
from explRL.clueboard import ClueBoardEnv
from explRL.llm_agent import LLMMember, runTurns
from explRL.llm_budget import estimateCost
from explRL.llm_grid import runAnyCell
from explRL.llm_runs import LLMConfig, cellRow, llmStats, runLLMCell
from explRL.multi import spawnRngs
from explRL.team_refs import ledgerArrays
from explRL.team_time import teamColumns

NAN = float('nan')
BOARD_KEYS = ('m', 'r', 'R', 'value', 'life', 'penalty')


def clueSetup(cfg, seed):
    "The board (settings from cfg.world) and the channel for one run."
    b = ClueBoardEnv(seed, cfg.N, **{k: v for k, v in cfg.world.items() if k in BOARD_KEYS})
    ch = dict(none=None, blackboard=ClueBoard(), hierarchical=ClueLead(cfg.N, cfg.world.get('passPooled', False)),
              messages=ClueMessages(cfg.N, 2, spawnRngs(seed + 2000, cfg.N)))[cfg.channel]
    return b, ch


def _shareToUse(b):
    "Mean time from the first true share of a clue the solver lacked to the assisted solve that used it."
    out = []
    for t, i, k, assisted in b.solveLog:
        if assisted:
            for j, hs in enumerate(b.tasks[k].holders):
                if i not in hs:
                    ts = [s[0] for s in b.shareLog if s[2] == k and s[3] == j and not s[6] and s[0] <= t]
                    if ts:
                        out.append(t - min(ts))
    return float(np.mean(out)) if out else NAN


def clueMetrics(cfg, b, hist, honest, T):
    "Welfare, team totals and the information metrics of one clue-board run."
    N, net, L, h = cfg.N, b.netByAgent(), ledgerArrays(b.ledger), list(honest)
    held = sum(len(hs) for s in b.tasks.values() for hs in s.holders)
    sharers = {s[1] for s in b.shareLog}
    rew = np.bincount(L['agent'], weights=L['reward'], minlength=N)
    grp = lambda ok: float(np.mean([rew[i] for i in h if ok(i)])) if any(ok(i) for i in h) else NAN
    return dict(welfare=float(net.sum()) / (N * T), welfareHonest=float(net[h].sum()) / (len(h) * T) if h else NAN,
                tEnd=float(T), solved=b.counts['solved'], opens=b.counts['opens'], wrong=b.counts['wrong'],
                late=b.counts['late'], guesses=sum(1 for s in b.submits if s[3] is not None and s[3] > 1),
                shares=b.counts['shares'], sharePerHeld=b.counts['shares'] / held if held else NAN,
                redundantShares=sum(s[5] for s in b.shareLog), falseShares=sum(s[6] for s in b.shareLog),
                assisted=float(np.mean([s[3] for s in b.solveLog])) if b.solveLog else NAN, shareToUse=_shareToUse(b),
                pooledShare=float(net.sum()) / (T * b.pooledBound()) if b.pooledBound() > 0 else NAN,
                sharerReward=grp(lambda i: i in sharers), nonSharerReward=grp(lambda i: i not in sharers),
                **teamColumns(L, np.array(b.netRows), 10.0, T))


def runClueNumpy(cfg, seed):
    "Rule agents (liars first) on one board, turn-limited as the LLM runs."
    b, ch = clueSetup(cfg, seed)
    T, rngs = cfg.turnsPerAgent(), spawnRngs(seed + 1000, cfg.N)
    members = [ClueLiar(ch, rngs[i]) if i < cfg.nLiar else
               RuleAgent(ch, rngs[i], cfg.world.get('share', 'needed'), cfg.world.get('guess', 0.0)) for i in range(cfg.N)]
    hist = runTurns(members, b, T)
    return [dict(episode=0, **clueMetrics(cfg, b, hist, range(cfg.nLiar, cfg.N), T))]


def runClue(cfg, seed, client):
    "LLM agents (scripted false-clue liars first) on one board -> (rows, turn log), as Phase 10's runLLM."
    T, nLLM = cfg.turnsPerAgent(), cfg.N - cfg.nLiar
    client.reserve(estimateCost(cfg.model, 'clue', nLLM, T))
    b, ch = clueSetup(cfg, seed)
    env = TextClueEnv(b, cfg.world.get('hint'), cfg.world.get('showOpen', False))
    liars = [ClueLiar(ch, r) for r in spawnRngs(seed + 3000, cfg.nLiar)]
    llm = [LLMMember(client, env, ch, cfg.memory, cfg.framing) for _ in range(nLLM)]
    hist = runTurns(liars + llm, b, T)
    row = dict(episode=0, **clueMetrics(cfg, b, hist, range(cfg.nLiar, cfg.N), T), **llmStats(llm))
    return [row], [dict(episode=0, **x) for m in llm for x in m.log]


def runClueAny(cell):
    "For runWorkload: clue cells to the clue runners, every other cell to runAnyCell."
    if cell['cfg'].get('task') != 'clue':
        return runAnyCell(cell)
    cfg = LLMConfig.fromDict(cell['cfg'])
    if cell['runner'] == 'numpy':
        return [cellRow(cell, cfg, r) for r in runClueNumpy(cfg, cell['seed'])]
    return runLLMCell(cell, run=runClue)

## Tests

In [ ]:
import numpy as np
from explRL.clue_runs import *
from explRL.llm_runs import LLMConfig
from explRL.llm_grid import llmCell, quickGrid

W = dict(m=3, r=1, R=105)
cfg = LLMConfig(task='clue', channel='blackboard', N=4, turns=40, world=W)
r = runClueNumpy(cfg, 900)[0]
for k in ('welfare', 'teamWelfare', 'teamRate_0', 'solved', 'guesses', 'shares', 'sharePerHeld', 'redundantShares',
          'assisted', 'shareToUse', 'pooledShare', 'sharerReward', 'nonSharerReward'):
    assert k in r, k
assert abs(r['teamWelfare'] - r['welfare'] * 4 * 40) < 1e-9 and r['shares'] > 0 and 0 < r['pooledShare'] < 1
assert abs(sum(r[f'teamRate_{k}'] for k in range(4)) * 10 - r['teamWelfare']) < 1e-9          # 40 turns = 4 periods of 10
assert runClueNumpy(cfg, 900) == runClueNumpy(cfg, 900)                     # deterministic per seed
none = runClueNumpy(LLMConfig(task='clue', channel='none', N=4, turns=40, world=W), 900)[0]
assert none['shares'] == 0 and none['solved'] == 0          # no channel, no guessing: no agent ever holds all clues (r m <= N (m - 1))
liar = runClueNumpy(LLMConfig(task='clue', channel='blackboard', N=4, nLiar=1, turns=40, world=W), 900)[0]
assert liar['falseShares'] > 0 and liar['welfareHonest'] != liar['welfare']

# hand-made: the share-to-use time and assisted solves on a scripted sequence
from explRL.clueboard import ClueBoardEnv
b = ClueBoardEnv(1, 3, m=3, r=1, R=105, life=100)
k = b.open[0]; s = b.tasks[k]; solver = 0
miss = [j for j, hs in enumerate(s.holders) if solver not in hs]
b.apply(solver, ('open', k), 0.0)
for t, j in enumerate(miss, start=1):
    b.apply(next(iter(s.holders[j])), ('share', (k, j, s.task.clue(j)[1])), float(t))
b.apply(solver, ('submit', s.task.x), 5.0)
from explRL.clue_runs import _shareToUse
assert b.solveLog[0][3] and _shareToUse(b) == np.mean([5.0 - t for t in range(1, len(miss) + 1)])

# runClueAny: clue cells to the clue runners, other cells unchanged
c = llmCell('clueNumpy', 'clueNumpy', 'clue', 4, 900, runner='numpy', task='clue', channel='blackboard', turns=10, world=W)
row = runClueAny(c)[0]
assert row['study'] == 'clueNumpy' and row['task'] == 'clue' and 'shares' in row
other = [x for x in quickGrid() if x['runner'] == 'numpy'][0]
assert runClueAny(other) == __import__('explRL.llm_grid', fromlist=['runAnyCell']).runAnyCell(other)

# LLM runs with the fake client: rows with the information and interface columns; turn log
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply
rows, log = runClue(LLMConfig(task='clue', channel='blackboard', N=4, turns=6, world=W), 900,
                    LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply)))
r = rows[0]
assert r['calls'] == 24 and len(log) == 24 and r['invalid'] < 24 and 'shares' in r and 'reportShare' in r
rl, _ = runClue(LLMConfig(task='clue', channel='messages', N=4, nLiar=1, turns=6, world=W), 900, LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply)))
assert rl[0]['calls'] == 18 and rl[0]['falseShares'] > 0
c = llmCell('clue', 'clue', 'clue', 4, 900, task='clue', channel='hierarchical', turns=4, world=W)
row = runClueAny(c)[0]
assert row['study'] == 'clue' and row['calls'] == 16

# Track A options travel in cfg.world (AC-2); without them the rows are unchanged
base = runClue(LLMConfig(task='clue', channel='hierarchical', N=4, turns=6, world=W), 900, LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply)))
again = runClue(LLMConfig(task='clue', channel='hierarchical', N=4, turns=6, world=dict(W)), 900, LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply)))
assert base == again
for opt in (dict(hint='follow'), dict(passPooled=True), dict(showOpen=True)):
    rows, log = runClue(LLMConfig(task='clue', channel='hierarchical', N=4, turns=6, world={**W, **opt}), 900,
                        LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply)))
    assert rows[0]['calls'] == 24
    if 'showOpen' in opt:
        assert all('Teammates working on:' in x['obs'] for x in log)
    b, ch = clueSetup(LLMConfig(task='clue', channel='hierarchical', N=4, world={**W, **opt}), 900)
    assert ch.passPooled == ('passPooled' in opt)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()